In [ ]:
import torch

from snrv import load_snrv

In [ ]:
model = load_snrv("snrv_50.pt")

In [ ]:
model = model.cpu()
model.device = "cpu"

In [ ]:
class CV(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def get_colvars(self, inps):

        phis = inps[:9]
        psis = inps[9:]

        sin_phi, cos_phi = torch.sin(phis), torch.cos(phis)
        sin_psi, cos_psi = torch.sin(psis), torch.cos(psis)
        feats = torch.stack([sin_phi, cos_phi, sin_psi, cos_psi], dim=1)

        return feats

    def forward(self, x):
        x = x.cpu()

        if x.dim() != 1:
            x = x.squeeze()
            
        feats = self.get_colvars(x)
        evecs = self.model.transform(feats.view(1, -1))
        CV = evecs[:, 1:2]
        return CV

model_output = CV(model).cpu()
model_output.eval()

torch.jit.script(model_output).save('srv_50.ptc')